In [3]:
"""
CSV 파일 벤치마크 도구
- pandas / pandas+dtype / polars(eager) / polars(lazy) 4가지 방법 비교
- 시간, 메모리, 결과값(페이지별 응답시간/디바이스 점유율/에러카운트)까지 확인

사용법: FILE_PATH만 원하는 CSV 파일 경로로 바꾸면 됩니다.
"""

import pandas as pd
import polars as pl
import time
import psutil
import os
import gc

# ===== 여기만 바꾸시면 됩니다 =====
FILE_PATH = "C:\Users\pyostry\workspace\aiffel\ai-data-bootcamp\D008\logs_raw.csv"
# ================================


def rss_mb():
    return psutil.Process(os.getpid()).memory_info().rss / 1024**2


def run_benchmark(name, func):
    gc.collect()
    before = rss_mb()
    start = time.perf_counter()
    result = func()
    elapsed = time.perf_counter() - start
    after = rss_mb()
    print(f"\n[{name}] 소요 시간: {elapsed:.2f}초, 메모리 증가량: {after - before:.1f} MB")
    return result


# ===== 방법 1: pandas (기본) =====
def method_pandas():
    df = pd.read_csv(FILE_PATH, parse_dates=["ts"])
    q1 = df.groupby("page")["response_ms"].mean().round(2)                                    # Q1: 페이지별 평균 응답시간
    q2 = df["device"].value_counts()                                                          # Q2: 디바이스별 총 트래픽 합계
    q3 = df[df["status_code"] >= 400].groupby("page").size().sort_values(ascending=False)     # Q3: 에러(400+) 페이지별 건수
    q4 = df.groupby("page")["response_ms"].median()                                            # Q4: 페이지별 응답시간 중앙값
    return q1, q2, q3, q4


# ===== 방법 2: pandas + dtype 최적화 =====
def method_pandas_dtype():
    df = pd.read_csv(
        FILE_PATH,
        dtype={"page": "category", "device": "category", "session_id": "int32", "bytes_sent": "int32"},
        parse_dates=["ts"]
    )
    q1 = df.groupby("page", observed=True)["response_ms"].mean().round(2)
    q2 = df["device"].value_counts()
    q3 = df[df["status_code"] >= 400].groupby("page", observed=True).size().sort_values(ascending=False)
    q4 = df.groupby("page", observed=True)["response_ms"].median()
    return q1, q2, q3, q4


# ===== 방법 3: polars (eager) =====
def method_polars_eager():
    df = pl.read_csv(FILE_PATH, try_parse_dates=True)
    q1 = df.group_by("page").agg(pl.col("response_ms").mean().round(2).alias("mean")).sort("page")
    q2 = df.group_by("device").agg(pl.len().alias("count")).sort("device")
    q3 = (
        df.filter(pl.col("status_code") >= 400)
        .group_by("page")
        .agg(pl.len().alias("n_errors"))
        .sort("n_errors", descending=True)
    )
    q4 = df.group_by("page").agg(pl.col("response_ms").median().alias("median")).sort("page")
    return q1, q2, q3, q4


# ===== 방법 4: polars (lazy) =====
def method_polars_lazy():
    lazy_df = pl.scan_csv(FILE_PATH, try_parse_dates=True)

    q1 = (
        lazy_df.group_by("page")
        .agg(pl.col("response_ms").mean().round(2).alias("mean"))
        .sort("page")
        .collect()
    )

    q2 = (
        lazy_df.group_by("device")
        .agg(pl.len().alias("count"))
        .sort("device")
        .collect()
    )

    q3 = (
        lazy_df.filter(pl.col("status_code") >= 400)
        .group_by("page")
        .agg(pl.len().alias("n_errors"))
        .sort("n_errors", descending=True)
        .collect()
    )

    q4 = (
        lazy_df.group_by("page")
        .agg(pl.col("response_ms").median().alias("median"))
        .sort("page")
        .collect()
    )
    return q1, q2, q3, q4


if __name__ == "__main__":
    with open(FILE_PATH, encoding="utf-8") as f:
        row_count = sum(1 for _ in f) - 1   # 헤더 줄 제외

    print("=" * 60)
    print(f"벤치마크 대상 파일: {FILE_PATH}")
    print(f"행 수 / 파일 크기: {row_count:,}행 / {os.path.getsize(FILE_PATH) / 1024**2:.1f} MB")
    print("=" * 60)

    results = {}
    results["pandas"] = run_benchmark("pandas (기본)", method_pandas)
    results["pandas_dtype"] = run_benchmark("pandas + dtype", method_pandas_dtype)
    results["polars_eager"] = run_benchmark("polars (eager)", method_polars_eager)
    results["polars_lazy"] = run_benchmark("polars (lazy)", method_polars_lazy)

    print("\n" + "=" * 60)
    print("pandas 기준 결과 (다른 방법들도 값이 같은지 비교해보세요)")
    print("=" * 60)
    q1, q2, q3, q4 = results["pandas"]
    print("\n[Q1] 페이지별 평균 응답시간")
    print(q1)
    print("\n[Q2] 디바이스별 총 트래픽 합계")
    print(q2)
    print("\n[Q3] status_code >= 400 인 페이지별 건수")
    print(q3)
    print("\n[Q4] 페이지별 응답시간 중앙값")
    print(q4)

SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (4235355005.py, line 17)